# Selecting dwarf galaxies from FastSpecFit

Fastspecfit has their catalogue split up among several fits files merged by top-level healpix ID. 
Since these are not easily queriable, this notebook takes the list of target IDs from the dwarf galaxy catalogue and selects all of the records from the various fastspecfit files which have matching target IDs. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from astropy.table import Table, vstack, hstack, join

In [ ]:
desi_dwarfs = Table.read("../data/stellar_mass_emline_dwarfs.fits")

In [ ]:
plt.hist(desi_dwarfs["program"])

In [ ]:
def read_fs_cat(program, i=0):
    
    t1 = Table.read(f"../data/fastspecfit/fastspec-iron-main-{program}-nside1-hp{i:02d}.fits", hdu=1)
    t2 = Table.read(f"../data/fastspecfit/fastspec-iron-main-{program}-nside1-hp{i:02d}.fits", hdu=2)
    t3 = Table.read(f"../data/fastspecfit/fastspec-iron-main-{program}-nside1-hp{i:02d}.fits", hdu=3)

    return hstack([t1, t2, t3])

# the loop

In [ ]:
fs_matched = None

for program in ["bright", "dark"]:
    for i in range(12):
        print(f"reading {program} {i}")

        df = read_fs_cat(program, i)

        print('matching')
        
        filt = np.isin(df["TARGETID_1"], desi_dwarfs["targetid"])
        if fs_matched is None:
            fs_matched = df[filt]
        else:
            fs_matched = vstack([fs_matched, df[filt]])

In [ ]:
fs_matched

In [ ]:
combined = join(desi_dwarfs[["targetid"]], fs_matched, keys_left=["targetid"], keys_right=["TARGETID_1"], join_type="left")

In [ ]:
assert np.all(fs_matched["TARGETID_1"] == fs_matched["TARGETID_2"])

In [ ]:
assert np.all(fs_matched["TARGETID_1"] == fs_matched["TARGETID_3"])

In [ ]:
combined

In [ ]:
combined[combined.colnames[:500]].write("../data/fastspecfit_dwarfs_part1.fits")

In [ ]:
combined[combined.colnames[500:]].write("../data/fastspecfit_dwarfs_part2.fits")